# Model training notebook
This notebook demonstrates preprocessing, training and evaluation of a RandomForest model to predict SpO2 (%) from Heart Rate (BPM). Use the `train_model.py` script for CLI training.

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
print('imports ok')

In [ ]:
# Load dataset (adjust path if needed)
df = pd.read_csv('synthetic_spo2_hr_1000.csv')
df.head()

In [ ]:
# Feature engineering
df['Heart Rate (BPM)'] = pd.to_numeric(df['Heart Rate (BPM)'], errors='coerce')
df['SpO2 (%)'] = pd.to_numeric(df['SpO2 (%)'], errors='coerce')
df['hr_roll_mean_5'] = df['Heart Rate (BPM)'].rolling(5, min_periods=1).mean()
df['hr_roll_std_5'] = df['Heart Rate (BPM)'].rolling(5, min_periods=1).std().fillna(0)
df['hr_diff'] = df['Heart Rate (BPM)'].diff().fillna(0)
df = df.dropna(subset=['SpO2 (%)'])
features = ['Heart Rate (BPM)','hr_roll_mean_5','hr_roll_std_5','hr_diff']
df[features + ['SpO2 (%)']].head()

In [ ]:
# Train / evaluate
X = df[features].values
y = df['SpO2 (%)'].values
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print('RMSE:', rmse)
print('R2:', r2_score(y_test,y_pred))

In [ ]:
# Save model artifact
joblib.dump({'model':model,'features':features}, 'IOT_Model_spo2_rf.joblib')
print('saved to IOT_Model_spo2_rf.joblib')